In [ ]:
%cd ../..

import os
import torch
from tqdm import tqdm
from omegaconf import OmegaConf
from glob import glob

from dinov2.inference import generate_embeddings, build_model, view_volume, crop_volume, is_HU

In [ ]:
import SimpleITK as sitk
import numpy as np

def load_mhd(path):
    image_obj = sitk.ReadImage(path)
    image = sitk.GetArrayFromImage(image_obj)
    spacing = image_obj.GetSpacing()
    spacing = np.array(spacing)[::-1]
    assert abs(spacing[2] - spacing[1]) < 0.001

    image = torch.from_numpy(image).float()
    image = image.clip(-1000, 1900)

    assert is_HU(image)

    return image, spacing

def crop_pos_mdh(path, posx, posy, posz): # posx, posy, posz, pad_xy, pad_z
    image_obj = sitk.ReadImage(path)
    image = sitk.GetArrayFromImage(image_obj)
    
    origin = np.array(image_obj.GetOrigin())
    spacing = np.array(image_obj.GetSpacing())
    direction = np.array(image_obj.GetDirection())
    direction = direction.reshape(3, 3)

    ijk = np.array([posx, posy, posz])

    print(image.shape)
    print(origin)
    print(spacing)
    print(direction)

In [ ]:
import pandas as pd

candidates_df = pd.read_csv("/scratch/VM/radio-foundation/datasets/LUNA16/candidates.csv")
candidatesv2_df = pd.read_csv("/scratch/VM/radio-foundation/datasets/LUNA16/candidates_V2.csv")
annotations_df = pd.read_csv("/scratch/VM/radio-foundation/datasets/LUNA16/annotations.csv")
annotations_df.head()

In [ ]:
sample_path = "/scratch/VM/radio-foundation/datasets/LUNA16/subset3/1.3.6.1.4.1.14519.5.2.1.6279.6001.274052674198758621258447180130.mhd"
sample_id = sample_path.split("/")[-1].split(".mhd")[0]

rows = candidates_df2[candidates_df2["seriesuid"] == sample_id]

#crop_pos_mdh(sample_path)

In [ ]:
candidates_df2.iloc[9]["seriesuid"]

In [ ]:
sample_path = "/scratch/VM/radio-foundation/datasets/LUNA16/subset3/1.3.6.1.4.1.14519.5.2.1.6279.6001.274052674198758621258447180130.mhd"
sample_id = sample_path.split("/")[-1].split(".")[0]
img, spacing = load_mhd(sample_path)
view_volume(crop_volume(img, k=9), spacing)
print(tuple(img.shape), spacing)

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_79999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
data_path = "/scratch/VM/radio-foundation/datasets/LUNA16"
output_path = "/scratch/VM/radio-foundation/cache/embeddings/LUNA16"

mhd_paths = glob(os.path.join(data_path, "**/*.mhd"), recursive=True)

In [ ]:
data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 224,
    patch_size = 14,
    device="cuda",
    block_size=64,
    autocast_ctx=autocast_ctx
)

for img_path in tqdm(mhd_paths):
    img_id = img_path.split("/")[-1].split(".")[0]

    img, spacing = load_mhd(sample_path)

    collated_features = generate_embeddings(
        img,
        model=model,
        **data_kwargs # type: ignore
    )

    torch.save(collated_features, os.path.join(output_path, f"{img_id}.pth"))
